# Outfit Maker - Polyvore pipeline (Colab)
Runs the ML half of the project on a free T4 GPU: embed -> train -> evaluate.

**Before you start:** Runtime -> Change runtime type -> T4 GPU. You need a Hugging Face account that has been granted access to the gated dataset `mvasil/polyvore-outfits` (you already have this).

## 1. Setup
Upload the `outfit-maker` folder to Drive (or `git clone` your repo), then adjust `REPO`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
REPO = '/content/drive/MyDrive/outfit-maker'   # <- where the repo lives
DATA = '/content/drive/MyDrive/outfit-maker-data'
%cd $REPO
!pip -q install -r ml/requirements.txt

In [ ]:
from huggingface_hub import notebook_login
notebook_login()   # paste YOUR token here; it stays in your Colab session

## 2. Embed (resumable)
Safe to re-run: ids already in the cache (including your old `embeddings_001.npz`) are skipped. Evaluation items (valid/test) are embedded first, then train items up to the cap. Copy your old chunk into the cache folder first if you want to reuse it.

In [ ]:
!python -m ml.embed_polyvore --data-dir $DATA/polyvore --cache-dir $DATA/polyvore_embedding_cache --max-train-items 20000

## 3. Train
Keeps the checkpoint with the best **validation** FITB accuracy. Never touches the test split.

In [ ]:
!python -m ml.train --data-dir $DATA/polyvore --cache-dir $DATA/polyvore_embedding_cache --out ml/checkpoints/compat.pt --epochs 25

## 4. Evaluate
Iterate with `--split valid`. Run `--split test` **once** at the very end, and copy the table into `docs/results.md`.

In [ ]:
!python -m ml.evaluate --data-dir $DATA/polyvore --cache-dir $DATA/polyvore_embedding_cache --checkpoint ml/checkpoints/compat.pt --split valid

In [ ]:
# FINAL, run once:
!python -m ml.evaluate --data-dir $DATA/polyvore --cache-dir $DATA/polyvore_embedding_cache --checkpoint ml/checkpoints/compat.pt --split test --out-json docs/test_results.json

## 5. Ablations (validation only)
With vs without type-specific heads, and embedding size 64 vs 128.

In [ ]:
for flag, name in [('', 'typed128'), ('--shared-head', 'shared128'), ('--out-dim 64', 'typed64')]:
    !python -m ml.train --data-dir $DATA/polyvore --cache-dir $DATA/polyvore_embedding_cache --out ml/checkpoints/ablate_{name}.pt $flag